In [ ]:
!rm -rf /kaggle/working/Real-ESRGAN
!git clone --depth 1 --branch master https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile inference.py inference/*.py encode/*.py audio/*.py inference/models/*.py
!cd /kaggle/working/Real-ESRGAN && python inference.py --help >/dev/null
!cd /kaggle/working/Real-ESRGAN && python -m audio.process --help >/dev/null
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E "(libx265|hevc_nvenc|libsvtav1|libaom-av1|av1_nvenc)" || true


In [ ]:
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime/ts_2.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"

START_TIME = 5 * 60 + 35
TEST_SECONDS = 15


In [ ]:
# 视频参数
VIDEO_ENHANCE = True

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2

# RIFE_FPS = 0 关闭 RIFE 并保持源帧率；>0 时必须 >= 输入视频源帧率。
RIFE_FPS = 60

# GPU：False=单 GPU(cuda:0)，True=双 GPU(cuda:0,1)。
DUAL_GPU = True

# BasicVSR++ 固定参数。
BVS_TILE_SIZE = 640
BVS_CLIP_LENGTH = 13
BVS_BATCH_SIZE = 1
BVS_STRENGTH = 1.0

# ===== 视频编码 =====
# 选择：HEVC CPU=libx265，HEVC GPU=hevc_nvenc，AV1 CPU=libsvtav1/libaom-av1，AV1 GPU=av1_nvenc。
VIDEO_CODEC = "hevc_nvenc"
ENCODE_GPU = 0

# HEVC 高质量参数
HEVC_CRF = 18                 # libx265：越低质量越高
HEVC_PRESET = "slow"         # libx265：slow 比 medium 更偏质量/压缩效率
HEVC_CQ = 18                  # hevc_nvenc：越低质量越高
HEVC_NVENC_PRESET = "p7"     # NVENC：P7 最高质量、最慢

# AV1 高质量参数
AV1_CRF = 18                  # libsvtav1/libaom-av1：越低质量越高
AV1_SVTAV1_PRESET = 4         # SVT-AV1：越低压缩效率越高、越慢；4 偏高质量
AV1_AOM_CPU_USED = 1          # libaom-av1：越低质量/压缩效率越高、越慢
AV1_CQ = 18                   # av1_nvenc：越低质量越高
AV1_NVENC_PRESET = "p7"      # NVENC：P7 最高质量、最慢


In [ ]:
# 音频参数
AUDIO_ENHANCE = True
AUDIO_CODEC = "aac"  # 增强开启时使用；关闭增强时自动 stream copy
AUDIO_BITRATE = "256k"


In [ ]:
import subprocess
import sys

effective_audio_codec = AUDIO_CODEC if AUDIO_ENHANCE else "copy"

if VIDEO_CODEC in {"libx265", "hevc_nvenc"}:
    CRF = HEVC_CRF
    PRESET = HEVC_PRESET
    CQ = HEVC_CQ
    NVENC_PRESET = HEVC_NVENC_PRESET
    SVTAV1_PRESET = AV1_SVTAV1_PRESET  # 未使用，仅满足统一 CLI 参数
    AOM_CPU_USED = AV1_AOM_CPU_USED     # 未使用，仅满足统一 CLI 参数
elif VIDEO_CODEC in {"libsvtav1", "libaom-av1", "av1_nvenc"}:
    CRF = AV1_CRF
    PRESET = HEVC_PRESET                # 未使用，仅满足统一 CLI 参数
    CQ = AV1_CQ
    NVENC_PRESET = AV1_NVENC_PRESET
    SVTAV1_PRESET = AV1_SVTAV1_PRESET
    AOM_CPU_USED = AV1_AOM_CPU_USED
else:
    raise ValueError(f"Unsupported VIDEO_CODEC in Notebook: {VIDEO_CODEC}")

if VIDEO_ENHANCE:
    command = [
        sys.executable, "/kaggle/working/Real-ESRGAN/inference.py",
        "--input", INPUT_VIDEO,
        "--output", OUTPUT_VIDEO,
        "--model", MODEL,
        "--model-path", MODEL_PATH,
        "--scale", str(SCALE),
        "--rife-fps", str(RIFE_FPS),
        "--gpu-ids", "0,1" if DUAL_GPU else "0",
        "--bvs-tile-size", str(BVS_TILE_SIZE),
        "--bvs-clip-length", str(BVS_CLIP_LENGTH),
        "--bvs-batch-size", str(BVS_BATCH_SIZE),
        "--bvs-strength", str(BVS_STRENGTH),
        "--video-codec", VIDEO_CODEC,
        "--crf", str(CRF),
        "--preset", PRESET,
        "--svtav1-preset", str(SVTAV1_PRESET),
        "--aom-cpu-used", str(AOM_CPU_USED),
        "--cq", str(CQ),
        "--nvenc-preset", NVENC_PRESET,
        "--encode-gpu", str(ENCODE_GPU),
        "--audio-codec", effective_audio_codec,
        "--audio-bitrate", AUDIO_BITRATE,
        "--start-time", str(START_TIME),
        "--test-seconds", str(TEST_SECONDS),
        "--ffmpeg-bin", "ffmpeg",
        "--ffprobe-bin", "ffprobe",
    ]
else:
    command = [
        sys.executable, "-m", "audio.process",
        "--input", INPUT_VIDEO,
        "--output", OUTPUT_VIDEO,
        "--audio-codec", effective_audio_codec,
        "--audio-bitrate", AUDIO_BITRATE,
        "--start-time", str(START_TIME),
        "--test-seconds", str(TEST_SECONDS),
        "--ffmpeg-bin", "ffmpeg",
        "--ffprobe-bin", "ffprobe",
    ]

if AUDIO_ENHANCE:
    command.append("--audio-enhance")

cwd = "/kaggle/working/Real-ESRGAN" if not VIDEO_ENHANCE else None
process = subprocess.Popen(
    command,
    cwd=cwd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
returncode = process.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)
